### Script per generare il file 'followers.csv' a partire dai tre file:
  - ai_all_followers.csv
  - chatgpt_all_followers.csv
  - ml_all_followers.csv

### Obiettivi:
1. 50000 archi.
3. Favorire archi che coinvolgono nodi di alto grado su entrambi i lati.
4. Includere **obbligatoriamente** i thread_user_pk (da ciascun file data seed):


In [5]:
from pathlib import Path
import pandas as pd
DATA_DIR = Path.cwd().parent.parent / 'data' / 'interim' 
DATA_DIR
PATH_AI = DATA_DIR / 'ai_all_followers.csv'
PATH_CHATGPT = DATA_DIR / 'chatgpt_all_followers.csv'
PATH_ML = DATA_DIR / 'ml_all_followers.csv'
PATH_OUTPUT = DATA_DIR.parent / 'processed' /'followers.csv'

In [ ]:
def main():
    INPUT_PATHS = [
        PATH_AI,
        PATH_CHATGPT,
        PATH_ML
    ]

    # -- 1) Carico e concateno tutti i CSV interni
    dfs = [pd.read_csv(p) for p in INPUT_PATHS]
    df_all = pd.concat(dfs, ignore_index=True)

    # -- 2) Suddivido in due DataFrame in base a "source" 
    # df_foll: tutte le righe con source == "followers"
    df_foll = df_all[df_all['source'] == 'followers'].copy()
    # df_fof: tutte le righe con source == "followers_of_followers"
    df_fof = df_all[df_all['source'] == 'followers_of_followers'].copy()

    # -- 3) Trovo l’insieme dei follower già presenti in df_foll --
    present_followers = set(df_foll['thread_follower_pk'].astype(str).tolist())

    # -- 4) Filtraggio condizionale di df_fof: 
    #     tengo solo le righe il cui thread_follower_pk è già in present_followers 
    df_fof_filtered = df_fof[
        df_fof['thread_follower_pk'].astype(str).isin(present_followers)
    ].copy()

    # -- 5) Ricombino: prima followers, poi i follower_of_followers filtrati 
    df = pd.concat([df_foll, df_fof_filtered], ignore_index=True)

    # -- 6) Rimuovo eventuali colonne di conteggio inutili (es. “count” nel nome) 
    df = df.drop(columns=[c for c in df.columns if "count" in c.lower()], errors="ignore")

    # Alias per chiarezza
    U = "thread_user_pk"
    F = "thread_follower_pk"

    # -- 7) Rimuovo duplicati (basati su coppia user-follower)
    df = df.drop_duplicates(subset=[U, F], keep='first')

    # --- 8) Calcolo statistiche finali
    total_edges = len(df)
    total_nodes = df[U].nunique() + df[F].nunique()
    ratio =  total_edges /total_nodes
    print(f"Edges: {total_edges}, Nodes: {total_nodes}, Ratio (edges/ nodes): {ratio:.3f}")

    # --- 9) Salvo il CSV finale
    df.to_csv(PATH_OUTPUT, index=False)
    print(f"👉 '{PATH_OUTPUT}' creato con successo.")

if __name__ == "__main__":
    main()


Edges: 65005, Nodes: 25087, Ratio (edges/ nodes): 2.591
👉 'c:\Users\pasqua\Desktop\univ\progettoasnm\Code\data_extraction\data\processed\followers.csv' creato con successo.
